# SpliceAImod — sharded run against a GCS bucket (no Google Drive)

Runtime → Change runtime type → **A100**. Run the cells top to bottom, then leave the tab open
(Pro) or close it (Pro+ background execution). Open this notebook in more runtimes to add
sessions: they coordinate through atomic claim objects in the bucket, so no two work on the
same shard and a lost session costs at most the shard in flight.

Bucket layout (`GCS_ROOT` below):

```
gs://var-annot-transfer-uk/
  shards/   shard_NNNN.vcf.gz + shards.json     <- upload once: gsutil -m cp shards/* gs://var-annot-transfer-uk/shards/
  out/      shard_NNNN.tsv.gz                    <- results (one object per shard, written atomically)
  state/    shard_NNNN.claim / .done, logs/      <- coordination; tiny
```

Everything else (install, reference download from NCBI, driver launch) is
`examples/colab_run.py`. Nothing touches Drive.

In [ ]:
from google.colab import auth
auth.authenticate_user()        # grants this VM your GCS access (one click)

In [ ]:
import os, subprocess, runpy
os.environ.update({
    "GCS_ROOT": "gs://var-annot-transfer-uk",
    "INSTALL": "1",
    "COLAB_AUTO_UNASSIGN": "1",              # driver releases this runtime when nothing is left to claim
    "ORIG_OUTPUT": "1",
    "VCF_OUTPUT": "1",
    # "ANNOTATION": "gencodev49",           # or MANEv1.4
    # "EXTRA_FLAGS": "--compile --conv-impl valid_nhwc",
    # "TORCH_BATCH": "256", "PRED_BATCH": "8192", "BATCH_WORKERS": "4",
})
if not os.path.isdir("/content/SpliceAImod"):
    subprocess.run(["git", "clone", "-q", "https://github.com/chundruv/SpliceAImod.git", "/content/SpliceAImod"], check=True)
_ = runpy.run_path("/content/SpliceAImod/examples/colab_run.py", run_name="__main__"); del _

## Monitor (re-run any time)

In [ ]:
import glob, subprocess
logs = sorted(glob.glob("/content/work/logs/driver_*.log"))
print(open(logs[-1]).read()[-2500:] if logs else "no log yet")
print("--- gpu worker:")
print(subprocess.run("tail -n 4 /content/work/tmp/*/GPU_0_w0.stderr 2>/dev/null", shell=True, capture_output=True, text=True).stdout)
!nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader
!pgrep -af "python[0-9.]* /content/[c]olab_driver.py" || echo "driver not running"
!echo "--- bucket:"; gsutil ls gs://var-annot-transfer-uk/state/ 2>/dev/null | sed 's#.*/##' | sort | tr '\n' ' '; echo
!echo "finished: $(gsutil ls gs://var-annot-transfer-uk/state/*.done 2>/dev/null | wc -l)   claimed: $(gsutil ls gs://var-annot-transfer-uk/state/*.claim 2>/dev/null | wc -l)   results: $(gsutil ls gs://var-annot-transfer-uk/out/ 2>/dev/null | wc -l)"

## Optional: block until this session's driver exits, then release the runtime

The driver already releases the runtime itself (`COLAB_AUTO_UNASSIGN=1`); this is the
belt-and-braces path for Pro+ background execution or a tab left open.

In [ ]:
import subprocess, time
while subprocess.run(["pgrep", "-f", "python[0-9.]* /content/[c]olab_driver.py"], capture_output=True).returncode == 0:
    time.sleep(60)
from google.colab import runtime; runtime.unassign()

## Afterwards, on your own machine

```
gsutil -m cp "gs://var-annot-transfer-uk/out/shard_*.tsv.gz" out/
python examples/merge_shard_tsvs.py out/ near_splice_1kb.spliceai.tsv.gz shards/shards.json
```

Per-session driver logs are mirrored to `gs://var-annot-transfer-uk/state/logs/`.